# Analyze results for retrieval refinement optimization

In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
from IPython.display import display

In [ ]:
pd.options.plotting.backend = "plotly"

create_images = True

baseline_runs = [
    "", "", "" # enter results files to analyse
]

comparing_runs = [
    "", "", "" # enter results files to analyse
]

metrics = [
    "context_precision","context_recall","nv_context_relevance","answer_relevancy",
    "nv_response_groundedness", "correctness","semantic_similarity",
    "trustworthiness","prompt_alignment"
]

metrics_to_compare = [
    "correctness", "semantic_similarity", "context_precision", "context_recall", "nv_context_relevance"
]

Flatten and combine all results from all runs into one dataframe

In [ ]:
baseline_data = []

for run_file in baseline_runs:
    with open(run_file, 'r') as f:
        run_data = json.load(f)
        run_name = f.name.split('/')[-1]
        for entry in run_data['results']:
            dataset_name = entry["dataset"]
            
            for result in entry["results"]:
                r = result.copy()
                r["dataset"] = dataset_name
                r["run"] = run_name
                baseline_data.append(r)

baseline_df = pd.DataFrame(baseline_data)

baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'context_precision'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'context_recall'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'answer_relevancy'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'semantic_similarity'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'trustworthiness'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'fact checking', 'nv_response_groundedness'] = float('nan')
baseline_df.loc[baseline_df['dataset'] == 'multiple choice', 'answer_relevancy'] = float('nan')

In [ ]:
comparing_data = []

for run_file in comparing_runs:
    with open(run_file, 'r') as f:
        run_data = json.load(f)
        run_name = f.name.split('/')[-1]
        for entry in run_data['results']:
            dataset_name = entry["dataset"]
            
            for result in entry["results"]:
                r = result.copy()
                r["dataset"] = dataset_name
                r["run"] = run_name
                comparing_data.append(r)

comparing_df = pd.DataFrame(comparing_data)

comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'context_precision'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'context_recall'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'answer_relevancy'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'semantic_similarity'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'trustworthiness'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'fact checking', 'nv_response_groundedness'] = float('nan')
comparing_df.loc[comparing_df['dataset'] == 'multiple choice', 'answer_relevancy'] = float('nan')

In [ ]:
baseline_detail = baseline_df.groupby('dataset')[metrics_to_compare].mean()
baseline_by_metric = baseline_df[metrics_to_compare].mean()

comparing_detail = comparing_df.groupby('dataset')[metrics_to_compare].mean()
comparing_by_metric = comparing_df[metrics_to_compare].mean()

baseline_by_dataset = baseline_df.groupby('dataset')[metrics_to_compare].mean().T.mean()
comparing_by_dataset = comparing_df.groupby('dataset')[metrics_to_compare].mean().T.mean()

baseline_by_run = baseline_df.groupby('run')[metrics_to_compare].mean()
comparing_by_run = comparing_df.groupby('run')[metrics_to_compare].mean()

## Visualizing results

In [ ]:
baseline_overall = baseline_df[metrics].mean().mean()
comparing_overall = comparing_df[metrics].mean().mean()

fig = go.Figure(go.Indicator(
    mode = "number+delta",
    value = np.round(comparing_overall,3),
    number = {'prefix': f"Overall changed from {np.round(baseline_overall,3)} to "},
    delta = {'position': "top", 'reference': np.round(baseline_overall,3)}
))

fig.show()

if create_images:
    pio.write_image(fig, 'overall-delta.pdf')

In [ ]:
fig = go.Figure()

baseline_text = np.round(baseline_by_metric.values.tolist(), 2)
comparing_text = np.round(comparing_by_metric.values.tolist(), 2)

fig.add_trace(
    go.Bar(
        x=baseline_by_metric.index.tolist(),
        y=baseline_by_metric.values.tolist(),
        name="baseline",
        marker_color="grey",
        text=baseline_text
    )
)

fig.add_trace(
    go.Bar(
        x=baseline_by_metric.index.tolist(),
        y=comparing_by_metric.values.tolist(),
        name="comparing",
        marker_color='#0EA4E3',
        text=comparing_text
    )
)

fig.update_layout(
    barmode='group',
    yaxis=dict(
        range=[0, 1],
        title="Mean score"
    ),
    xaxis=dict(
        title="Metric"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'metric-comparison-bar.pdf')

In [ ]:
fig = go.Figure()

baseline_text = np.round(baseline_by_dataset.values.tolist(), 2)
comparing_text = np.round(comparing_by_dataset.values.tolist(), 2)

fig.add_trace(
    go.Bar(
        x=baseline_by_dataset.index.tolist(),
        y=baseline_by_dataset.values.tolist(),
        name="baseline",
        marker_color="grey",
        text=baseline_text
    )
)

fig.add_trace(
    go.Bar(
        x=baseline_by_dataset.index.tolist(),
        y=comparing_by_dataset.values.tolist(),
        name="comparing",
        marker_color='#0EA4E3',
        text=comparing_text
    )
)


fig.update_layout(
    barmode='group',
    yaxis=dict(
        range=[0, 1],
        title="Mean score"
    ),
    xaxis=dict(
        title="Metric"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'dataset-comparison-bar.pdf')

In [ ]:
metrics_diffs = comparing_by_metric - baseline_by_metric
metrics_percentage_diff = np.round(metrics_diffs / baseline_by_metric * 100, 2)

fig = go.Figure()

for metric in metrics_diffs.index:
    y = metrics_diffs[metric]
    color = "#FF7B52" if y <= 0.0 else "#56D752"
    fig.add_trace(
        go.Bar(x=[metric], y=[y], marker_color=color, text=f"{np.round(y, 4)}<br>({metrics_percentage_diff[metric]}%)")
    )

fig.update_layout(
    yaxis=dict(
        range=[-0.4, 0.4],
        title="Delta"
    ),
    xaxis=dict(
        title="Considered metrics"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'metrics-delta-bar.pdf')

In [ ]:
dataset_diffs = comparing_by_dataset - baseline_by_dataset
dataset_percentage_diffs = np.round(dataset_diffs / baseline_by_dataset * 100, 2)

fig = go.Figure()

for dataset in dataset_diffs.index:
    y = dataset_diffs[dataset]
    color = "#FF7B52" if y <= 0.0 else "#56D752"
    fig.add_trace(
        go.Bar(x=[dataset], y=[y], marker_color=color, text=f"{np.round(y, 4)}<br>({dataset_percentage_diffs[dataset]}%)")
    )

fig.update_layout(
    yaxis=dict(
        range=[-0.4, 0.4],
        title="Delta"
    ),
    xaxis=dict(
        title="Dataset"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'dataset-delta-bar.pdf')

In [ ]:
baseline_mean_correctness = baseline_df['correctness'].mean()
baseline_mean_sem_sim = baseline_df['semantic_similarity'].mean()
baseline_mean_combined = (baseline_mean_correctness+baseline_mean_sem_sim)/2

comparing_mean_correctness = comparing_df['correctness'].mean()
comparing_mean_sem_sim = comparing_df['semantic_similarity'].mean()
comparing_mean_combined = (comparing_mean_correctness+comparing_mean_sem_sim)/2

fig = go.Figure()

baseline_text = [
    np.round(baseline_mean_combined, 2)
]
comparing_text = [
    np.round(comparing_mean_combined, 2)
]

fig.add_trace(
    go.Bar(
        x=['baseline'],
        y=[baseline_mean_combined],
        name="baseline",
        marker_color="grey",
        text=baseline_text
    )
)

fig.add_trace(
    go.Bar(
        x=['comparing'],
        y=[comparing_mean_combined],
        name="comparing",
        marker_color='#0EA4E3',
        text=comparing_text
    )
)

fig.add_hline(y=0.7, line_dash="dash", line_color="red", line_width=2)


fig.update_layout(
    barmode='group',
    yaxis=dict(
        range=[0.55, 0.8],
        title=""
    ),
    xaxis=dict(
        title="Combined mean<br>(correctness & semantic_similarity)"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=550,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'combined-correctness.pdf')

Die Antwortkorrektheit als Mittelwert von correctness und semantic_similarity ist in 3 Durchläufen konstant über 0.7.
Das Primärziel ist somit erreicht.

In [ ]:
import plotly.express as px

lengths = [len(context) for context in comparing_df['retrieved_contexts']]
correctness_deltas = comparing_df['correctness']-baseline_df['correctness']
context_recall_deltas = comparing_df['context_recall']-baseline_df['context_recall']
context_relevance_deltas = comparing_df['nv_context_relevance']-baseline_df['nv_context_relevance']

used_refinement = [length > 4 for length in lengths]
increased_correctness = [delta > 0 for delta in correctness_deltas]
increased_context_recall = [delta > 0 for delta in context_recall_deltas]
increased_context_relevance = [delta > 0 for delta in context_relevance_deltas]

data = {
    "context erweitert": used_refinement,
    "correctness gesteigert": increased_correctness,
    "context_recall gesteigert": increased_context_recall,
    "nv_context_relevance gesteigert": increased_context_relevance
}

corr_df = pd.DataFrame(data)

corr_matrix = np.round(corr_df.corr(), 4)

fig = px.imshow(corr_matrix, text_auto=True)

fig.update_layout(
    title="Pairwise correlations of binary features",
    font=dict(
        size=16,
    ),
)

fig.show()

if create_images:
    pio.write_image(fig, 'binary-correlation.png', scale=5)

In [ ]:
x = np.array(comparing_df['context_recall'].values)
y = np.array(comparing_df['nv_context_relevance'].values)

mask = ~np.isnan(x) & ~np.isnan(y)
print('Correlation coefficient: context_recall / nv_context_relevance:')
print(np.corrcoef(x[mask], y[mask])[0][1])

x = np.array(comparing_df['correctness'].values)
y = np.array(comparing_df['nv_context_relevance'].values)

mask = ~np.isnan(x) & ~np.isnan(y)

print('Correlation coefficient: nv_context_relevance / correctness:')
print(np.corrcoef(x[mask], y[mask])[0][1])

x = np.array(comparing_df['correctness'].values)
y = np.array(comparing_df['context_recall'].values)

mask = ~np.isnan(x) & ~np.isnan(y)

print('Correlation coefficient: context_recall / correctness:')
print(np.corrcoef(x[mask], y[mask])[0][1])

In [ ]:
baseline_mean_generation_time = baseline_df['generation_time'].mean()
comparing_mean_generation_time = comparing_df['generation_time'].mean()

fig = go.Figure()

baseline_text = [
    np.round(baseline_mean_generation_time, 2)
]
comparing_text = [
    np.round(comparing_mean_generation_time, 2)
]

x = ["baseline", "comparing"]

fig.add_trace(
    go.Bar(
        x=['baseline'],
        y=[baseline_mean_generation_time],
        marker_color="grey",
        text=baseline_text
    )
)

fig.add_trace(
    go.Bar(
        x=['with refinement option'],
        y=[comparing_mean_generation_time],
        marker_color='#0EA4E3',
        text=comparing_text
    )
)

fig.update_layout(
    barmode='group',
    yaxis=dict(
        range=[0, 80],
        title=""
    ),
    xaxis=dict(
        title="Mean response latency (in s)"
    ),
    template="plotly_white",
    showlegend=False,
    height=500,
    width=550,
    font=dict(
        size=16,
    )
)

fig.show()

if create_images:
    pio.write_image(fig, 'response-time.pdf')

In [ ]:
detail = comparing_df.groupby('dataset')[metrics].mean()

text_matrix = detail.round(2).astype(str)
text_matrix = text_matrix.replace("nan", "n.a.")

deatil_fig = go.Figure(data=go.Heatmap(
    z=detail,
    x=detail.columns,
    y=detail.index,
    text=text_matrix,
    zmin=0.0,
    zmax=1.0,
    texttemplate="%{text}",
    colorscale=[(0.00, "#FF7B52"), (0.5, "#FF7B52"), 
                (0.5, "#FFC44F"), (0.7, "#FFC44F"),
                (0.7, "#F1FF53"), (0.9, "#F1FF53"),
                (0.9, "#56D752"),  (1.00, "#56D752")]
))

deatil_fig.update_layout(
    xaxis_title="Metrics",
    yaxis_title="Datasets",
    title="Mean score per dataset and metric",
    xaxis_nticks=len(detail.columns),
    yaxis_nticks=len(detail.index),
    font=dict(
        size=16,
    ),

)
    
deatil_fig.show()

if create_images:
    pio.write_image(deatil_fig, 'detail-scores.png', scale=5)

In [ ]:
results_by_metric = comparing_df[metrics].mean()

results_by_metric.rename("mean score", inplace=True)

display(results_by_metric)

mt_means_fig = go.Figure()

for metric in results_by_metric.index:
    y = results_by_metric[metric]
    color = "#FF7B52" if y <= 0.5 else "#FFC44F" if (y <= 0.7 and y > 0.5) else "#F1FF53" if (y <= 0.9 and y > 0.7) else "#56D752"
    mt_means_fig.add_trace(
        go.Bar(x=[metric], y=[results_by_metric[metric]], marker_color=color, text=np.round(y, 2))
    )

mt_means_fig.update_layout(
    xaxis_title='Metric',
    yaxis_title='Mean score',
    showlegend=False,
    height=500,
    width=700,
    font=dict(
        size=16,
    ),
)

mt_means_fig.show()

if create_images:
    pio.write_image(mt_means_fig, 'metric-means-bar.pdf')